# Phase 4 — The sub-sampling PLL (steps 46–58)

Sampling the VCO directly with the reference turns timing error into a large voltage. There is no divider in the loop, and the phase-detector and charge-pump noise are **not** multiplied by N² (Gao, Klumperink, Bohsali, Nauta, JSSC 2009). This phase builds the SSPLL from the sampling theorem up, derives every gain and noise term for our 50 MHz × 32 = 1.6 GHz design, and simulates acquisition with the dead-zone FLL — the exact architecture of the proposal.

**How to use this notebook.** Every step follows the same loop: **What happens** (the observation) →
**How it happens** (the mechanism and the equations, in Razavi's notation) → **Why it happens** (the physical
reason) → an interactive figure → **Try it** (what to change and what to look for) → **Interview questions**
(click a question to open the answer; try to answer it aloud first).

Notation (Razavi, *RF Microelectronics* ch. 9 and *Design of CMOS PLLs*): $\phi_{in}$, $\phi_{out}$ input and output
phase; $\omega_{out} = \omega_0 + K_{VCO} V_{cont}$; $K_{PD}$ phase-detector gain; $I_p$ charge-pump current;
$R_1, C_1, C_2$ loop filter; $N$ divide ratio; $\omega_n$, $\zeta$ natural frequency and damping;
$H(s) = \Phi_{out}/\Phi_{in}$. Frequencies $f$ in Hz, angular $\omega = 2\pi f$ in rad/s.
Design numbers are the submitted proposal's: $f_{REF} = 50$ MHz, $N = 32$, $f_{out} = 1.6$ GHz, $f_c \approx 2.5$ MHz.

## Contents

- [Step 46 — Sampling fundamentals: sample-and-hold, aliasing, sub-sampling 1.6 GHz at 50 MHz](#step-46)
- [Step 47 — Why the sample is DC at lock; sample ∝ timing error; K_t and K_PD](#step-47)
- [Step 48 — Phase-domain model: virtual ×N on the reference, no ÷N in the feedback](#step-48)
- [Step 49 — Sampler kT/C noise: single-ended vs differential, numbers for 54 fF and 100 fF](#step-49)
- [Step 50 — Pulsed-Gm charge pump: why the gain is reduced, β = 2·A·g_m·τ/T_REF, noise ∝ T/τ, dump node](#step-50)
- [Step 51 — Loop dynamics with the hold: ZOH sinc, T/2 delay, charge-sharing pole, phase-margin cost](#step-51)
- [Step 52 — Acquisition: periodic SSPD characteristic, false lock to N ± k, dead-zone FLL, hand-over](#step-52)
- [Step 53 — What the SSPLL solves (request 8)](#step-53)
- [Step 54 — Limit: narrow capture range, false lock, slow relock after a disturbance (Hsu, Geng)](#step-54)
- [Step 55 — Limit: sampler spurs (load modulation, charge injection, charge sharing); REF noise still ×N²; ring noise beyond f_c](#step-55)
- [Step 56 — Limit: sinusoidal gain non-linearity, ambiguity beyond ±π/2, ring supply sensitivity](#step-56)
- [Step 57 — Fixes: dummy samplers, isolation buffers, tracking window, low-noise REF buffer, combined PD, QSSPD, FFNC](#step-57)
- [Step 58 — Lab: SSPLL lock, explained waveform by waveform](#step-58)

*13 steps, 112 interview questions.*

In [ ]:
# setup - run this cell first (Shift+Enter). Figures are interactive: drag the slider under each plot,
# hover for values, drag to zoom, double-click to reset.
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sspll_learn as sl
from sspll_learn import P, show, sweep, bode, kT
np.set_printoptions(precision=4, suppress=True)
print("ready - proposal design point:", f"{P['fref']/1e6:.0f} MHz x {P['N']} = {P['f0']/1e9:.2f} GHz")

<a id='step-46'></a>
## Step 46 — Sampling fundamentals: sample-and-hold, aliasing, sub-sampling 1.6 GHz at 50 MHz

*Roadmap step 46: spectrum plot*

### What happens
Sampling a signal at rate $f_s$ cannot distinguish frequencies that differ by multiples of $f_s$: a tone at $f$ appears at the
**alias** $|f - kf_s|$ closest to DC. Sub-sampling deliberately samples a high-frequency signal far below Nyquist.

### How
Sample $v(t) = A\sin(2\pi f_{VCO}t + \phi)$ at $t_k = kT_{REF}$:
$$v_k = A\sin\!\big(2\pi f_{VCO}kT_{REF} + \phi\big) = A\sin\!\big(2\pi (f_{VCO}-Nf_{REF})kT_{REF} + \phi\big)$$
because $2\pi Nf_{REF}kT_{REF} = 2\pi Nk$ is a whole number of cycles. So the samples are a sine at the **beat frequency**
$f_{VCO}-Nf_{REF}$. When $f_{VCO} = 32\times50\text{ MHz} = 1.6$ GHz exactly, the beat is zero and **every sample has the same value**
$A\sin\phi$ — a DC level that depends only on the phase.

### Why it works as a phase detector
The sampler is a mixer with the reference's harmonics: the $N$-th harmonic of the sampling pulse train down-converts the VCO to DC.
No divider is needed to "bring the VCO down" to $f_{REF}$.

In [ ]:
fref, N = 50e6, 32
t = np.linspace(0, 12/fref, 6000); k = np.arange(13)
def frame(df_MHz):
    fv = N*fref + df_MHz*1e6
    v = np.sin(2*np.pi*fv*t + 0.4)
    vk = np.sin(2*np.pi*fv*k/fref + 0.4)
    beat = np.sin(2*np.pi*df_MHz*1e6*t + 0.4)
    return ([dict(x=t*1e9, y=v, name="VCO waveform", line=dict(width=0.6)),
             dict(x=k/fref*1e9, y=vk, mode="markers", marker=dict(size=10), name="samples at REF edges"),
             dict(x=t*1e9, y=beat, name="beat |f_VCO - 32 f_REF|", line=dict(dash="dash"))],
            f"f_VCO = {fv/1e9:.4f} GHz: samples form a {abs(df_MHz):.1f} MHz beat" + (" -> DC (locked)" if df_MHz == 0 else ""))
fig = sweep([-4, -2, -1, 0, 1, 2, 4], frame, "f_VCO - 1.6 GHz (MHz)", active=4,
            layout=lambda f: f.update_xaxes(title="time (ns)").update_yaxes(range=[-1.3, 1.3]))
show(fig, 440)

### Interview questions — step 46 (9)

<details><summary><b>Q1.</b> What is sub-sampling?</summary>

Sampling a band-pass signal at a rate below twice its highest frequency (but adequate for its bandwidth); the signal aliases to a lower frequency. In an SSPLL the VCO aliases to DC at lock.

</details>

<details><summary><b>Q2.</b> Why are the samples constant at lock?</summary>

$f_{VCO} = Nf_{REF}$ exactly, so between two samples the VCO completes exactly N cycles and is at the same phase.

</details>

<details><summary><b>Q3.</b> What frequency do the samples show when the VCO is off by Δf?</summary>

The beat $|f_{VCO} - kf_{REF}|$ for the nearest integer k; e.g. 1.601 GHz sampled at 50 MHz shows 1 MHz.

</details>

<details><summary><b>Q4.</b> Why is a sampler equivalent to a mixer here?</summary>

The sampling pulse train contains all harmonics of $f_{REF}$; the 32nd harmonic multiplies the VCO and produces a DC term proportional to $\sin\phi$.

</details>

<details><summary><b>Q5.</b> What is aliasing and why is it usually avoided?</summary>

Frequency components above Nyquist fold into the band; in data conversion it corrupts the signal, so anti-alias filters are used. In the SSPLL the folding is the mechanism; its side effect is that VCO noise at offsets around multiples of $f_{REF}$ also folds (relevant for FFNC, Phase 6).

</details>

<details><summary><b>Q6.</b> What sampling bandwidth must the SSPD switch have?</summary>

Enough to track a 1.6 GHz waveform: the switch on-resistance × sampling capacitor must be well below the VCO period (track time constant ≪ 100 ps) to avoid gain loss and phase shift.

</details>

<details><summary><b>Q7.</b> Does the SSPD care about f_REF duty cycle?</summary>

Only the sampling edge matters; duty affects the track/hold time available.

</details>

<details><summary><b>Q8.</b> What is the 'aperture jitter' of a sampler and why does it matter here?</summary>

Uncertainty of the sampling instant; it adds directly to the measured phase error, like reference jitter (×N² at the output).

</details>

<details><summary><b>Q9.</b> How many samples per VCO period does the SSPD take?</summary>

One sample every 32 VCO periods — far from Nyquist for the waveform, but the information needed (phase at one point) is captured.

</details>

<a id='step-47'></a>
## Step 47 — Why the sample is DC at lock; sample ∝ timing error; K_t and K_PD

*Roadmap step 47: K_t = slope at the zero crossing; K = A (sine) or edge slope (ring)*

### What happens
At lock the loop drives the sampled value to the zero crossing of the VCO waveform. A small timing error $\Delta t$ between the
reference edge and the VCO zero crossing then gives a proportional sample.

### How
$$v_{sam} = A\sin(\omega_{VCO}\Delta t)\approx \underbrace{A\,\omega_{VCO}}_{K_t\ [\text{V/s}]}\Delta t = \underbrace{A}_{K_{PD}\ [\text{V/rad}]}\Delta\phi_{VCO}.$$
For a ring oscillator the waveform is closer to a trapezoid; the gain is the **slope at the crossing**: $K_t = SR$ (slew rate, V/s) and
$K_{PD} = SR/\omega_{VCO}$ (V per VCO radian). Nagam & Kinget write $K_{t,SSPD}$ (V/s) and $K_{\phi,SSPD} = K_{t,SSPD}/(2\pi f_{out})$.

**Numbers (assumed amplitude A = 0.5 V at 1.6 GHz):** $K_{PD} = 0.5$ V/rad, $K_t = 0.5\cdot2\pi\cdot1.6\times10^9 = 5.0$ mV/ps.
Compare a PFD/CP: $I_p/(2\pi N)$ = 100 µA/(2π·32) = 0.50 µA per VCO radian into the filter.

### Why it matters
The SSPD gain is large and referred directly to VCO phase — so its own noise, divided by this gain, is tiny in phase terms. It is also
**amplitude-dependent** (A or SR changes with supply and corner), which moves the loop gain (step 56).

In [ ]:
fv = 1.6e9; tt = np.linspace(-0.4/fv, 0.4/fv, 400)
def frame(dt_ps):
    sine = 0.5*np.sin(2*np.pi*fv*tt)
    ring = 0.5*np.tanh(3*np.sin(2*np.pi*fv*tt))/np.tanh(3)          # sharper, ring-like waveform
    vs_s = 0.5*np.sin(2*np.pi*fv*dt_ps*1e-12); vs_r = 0.5*np.tanh(3*np.sin(2*np.pi*fv*dt_ps*1e-12))/np.tanh(3)
    return ([dict(x=tt*1e12, y=sine, name="sine, A = 0.5 V"), dict(x=tt*1e12, y=ring, name="ring-like waveform"),
             dict(x=[dt_ps, dt_ps], y=[vs_s, vs_r], mode="markers", marker=dict(size=11), name="samples at REF edge")],
            f"timing error {dt_ps} ps: sine sample {vs_s*1e3:.1f} mV (K_t = {0.5*2*np.pi*fv*1e-12*1e3:.2f} mV/ps), "
            f"ring sample {vs_r*1e3:.1f} mV (steeper slope, saturates earlier)")
fig = sweep([-60, -20, -5, 0, 5, 20, 60, 120], frame, "timing error (ps)", active=4,
            layout=lambda f: f.update_xaxes(title="time around the VCO zero crossing (ps)").update_yaxes(title="V"))
show(fig, 430)

### Interview questions — step 47 (9)

<details><summary><b>Q1.</b> Derive the SSPD gain.</summary>

$v = A\sin(\omega_{VCO}\Delta t) \approx A\omega_{VCO}\Delta t$; per VCO radian $K_{PD} = A$; per second $K_t = A\omega_{VCO}$.

</details>

<details><summary><b>Q2.</b> Why does the SSPLL lock at a zero crossing?</summary>

That is where the sample is zero for zero CP current (no correction needed) and where the slope (gain) is maximum, so the equilibrium is stable and linear.

</details>

<details><summary><b>Q3.</b> Which zero crossing — rising or falling?</summary>

The one with the slope sign that gives negative feedback for the chosen CP polarity; the other crossing is an unstable equilibrium.

</details>

<details><summary><b>Q4.</b> How do you define the SSPD gain for a ring oscillator?</summary>

By the slope (slew rate) of the sampled waveform at the crossing: $K_t = dv/dt$; it depends on the ring's swing and rise time.

</details>

<details><summary><b>Q5.</b> How does the SSPD gain compare with a CP-PLL detector referred to the VCO?</summary>

CP: $I_p/(2\pi N)$ A per VCO radian (shrinks with N). SSPD: $A$ V per VCO radian, then $g_m$ (and duty) converts it to current — orders of magnitude larger before scaling.

</details>

<details><summary><b>Q6.</b> Why is the SSPD gain amplitude-dependent and is that a problem?</summary>

It is the waveform slope; the amplitude varies with supply, corner and frequency, so loop gain (bandwidth, PM) varies; designs limit the variation or accept it with margin.

</details>

<details><summary><b>Q7.</b> What is the timing resolution needed to read 1 mV of SSPD output?</summary>

$1\,mV / 5.0\,mV/ps \approx 0.2$ ps — the SSPD is a very sensitive timing detector.

</details>

<details><summary><b>Q8.</b> What limits the linear range of the SSPD?</summary>

The sine's curvature: linear only for $|\Delta\phi| \ll \pi/2$, i.e. about ±40 ps of a 625 ps period for <10% gain error; beyond ±π/2 the slope reverses.

</details>

<details><summary><b>Q9.</b> Relation between K_t and K_phi in Nagam & Kinget?</summary>

$K_{\phi,SSPD} = K_{t,SSPD}/(2\pi f_{out})$; the NCB gain must be $1/K_t$ (s/V), equivalently $1/K_\phi$ (rad/V).

</details>

<a id='step-48'></a>
## Step 48 — Phase-domain model: virtual ×N on the reference, no ÷N in the feedback

*Roadmap step 48: why PD/CP noise skips N² but reference noise does not*

### What happens
The SSPD compares the VCO phase with the reference phase **multiplied by N** (in VCO radians, the reference edge advances
$2\pi N$ per period). There is no divider in the feedback.

### How
Let $\beta$ be the SSPD+CP gain in amperes per VCO radian (step 50: $\beta = 2A g_m\tau/T_{REF}$). Then
$$G(s) = \beta\,Z(s)\,\frac{K_{VCO}}{s},\qquad \Phi_{out} = \frac{G}{1+G}\big(N\Phi_{REF} + \Phi_{n,SSPD}\big) + \frac{1}{1+G}\Phi_{VCO}.$$
Compare the CP-PLL: $G = \frac{I_p}{2\pi N}Z\frac{K_{VCO}}{s}$ and $\Phi_{out} = \frac{NG}{1+G}(\Phi_{REF} + \Phi_{n,PFD/CP})+\dots$

**Noise referral.** An SSPD voltage noise $v_n$ appears as $\Phi_{n,SSPD} = v_n/A$ **VCO** radians; it reaches the output with gain 1
in band. A PFD/CP noise current $i_n$ appears as $2\pi i_n/I_p$ **reference** radians and reaches the output ×N. Reference
jitter is a time error; any loop converts it to $N\times$ more VCO radians — so reference noise is ×N² in both.

### Why it matters
The detector's noise floor drops by roughly $20\log N$ (30 dB for N = 32), leaving the reference (crystal + buffer) as the in-band
limit: −152 dBc/Hz + 30.1 = −121.9 dBc/Hz at 1.6 GHz in our budget.

In [ ]:
import sys; sys.path.insert(0, "../models")
import pllmodel as pm
f = pm.freq_axis(n=500)
def frame(fc_MHz):
    H = pm.loop_H(f, fc_MHz*1e6, 50)
    return ([dict(x=f, y=20*np.log10(abs(H)) + 20*np.log10(32), name="REF -> out: N H (+30.1 dB in band)"),
             dict(x=f, y=20*np.log10(abs(H)), name="SSPD noise (VCO rad) -> out: H"),
             dict(x=f, y=20*np.log10(abs(H)) + 20*np.log10(32), name="CP-PLL PFD/CP noise (REF rad) -> out: N H", line=dict(dash="dot")),
             dict(x=f, y=20*np.log10(abs(1-H)), name="VCO -> out: 1 - H")],
            f"noise transfer functions, f_c = {fc_MHz} MHz (PM 50 deg)")
fig = sweep([1.0, 2.5, 5.0], frame, "f_c (MHz)", active=1,
            layout=lambda f: f.update_xaxes(type="log", title="offset (Hz)").update_yaxes(title="dB", range=[-60, 40]))
show(fig, 440)

### Interview questions — step 48 (8)

<details><summary><b>Q1.</b> Why is PD noise not multiplied by N² in an SSPLL?</summary>

The SSPD measures the VCO phase directly (gain A per VCO radian); its noise referred to the output is $v_n/A$ VCO radians with unity in-band gain. A PFD measures divided phase, so its noise referred to the output is multiplied by N.

</details>

<details><summary><b>Q2.</b> Is the reference noise still multiplied by N²?</summary>

Yes: a reference timing error $\Delta t$ is $2\pi f_{out}\Delta t = N\cdot2\pi f_{REF}\Delta t$ VCO radians; the loop reproduces it.

</details>

<details><summary><b>Q3.</b> What is the loop gain of an SSPLL?</summary>

$G = \beta Z(s) K_{VCO}/s$ with $\beta$ in A per VCO radian; formally the CP-PLL expression with $I_p/(2\pi N)$ replaced by $\beta$.

</details>

<details><summary><b>Q4.</b> Where does the 'virtual multiplication by N' come from?</summary>

Each reference period corresponds to N VCO cycles, so the reference phase expressed in VCO radians is $N\phi_{REF}$; the SSPD compares $\phi_{VCO}$ with that.

</details>

<details><summary><b>Q5.</b> What is the in-band floor of our SSPLL according to the budget?</summary>

Reference: −152 + 30.1 = −121.9 dBc/Hz; SSPD+CP: −130 dBc/Hz (typ). The reference dominates in band.

</details>

<details><summary><b>Q6.</b> Does the divider noise matter in an SSPLL?</summary>

Not in lock: the divider (FLL) is outside the main loop and its PFD is in its dead zone.

</details>

<details><summary><b>Q7.</b> Why do SSPLLs reach −120 to −126 dBc/Hz in-band at 2 GHz (Gao) while CP-PLLs rarely do?</summary>

No N² on the detector; the remaining floor is the reference (×N²) and the sampler/Gm noise referred by a large gain A.

</details>

<details><summary><b>Q8.</b> In which units must you be careful when mixing SSPLL and CP-PLL formulas?</summary>

SSPLL gains are per VCO radian, CP-PLL gains per reference radian (factor N), and $K_{VCO}$ is in rad/s/V.

</details>

<a id='step-49'></a>
## Step 49 — Sampler kT/C noise: single-ended vs differential, numbers for 54 fF and 100 fF

*Roadmap step 49: noise table*

### What happens
Each time the sampling switch opens, the capacitor freezes a random thermal-noise voltage of variance $kT/C$ (independent of the
switch resistance: a larger R gives more noise density but a narrower bandwidth).

### How — referred to output phase noise
One sample per reference period: $\sigma_v^2 = kT/C$ (single-ended). In VCO radians $\sigma_\phi^2 = \sigma_v^2/A^2$. A white sequence held
for $T_{REF}$ has a one-sided PSD of $2\sigma^2/f_{REF}$ at low frequency, so (with $\mathcal L = S_\phi/2$)
$$\mathcal L_{SSPD} = \frac{kT}{C\,A^2 f_{REF}}\ (\text{single-ended}),\qquad
\mathcal L_{SSPD,diff} = \frac{2kT/C}{(2A)^2 f_{REF}} = \frac{kT}{2CA^2f_{REF}}\ (\text{differential, 3 dB better}).$$
This in-band noise is **not** multiplied by N.

### Why the numbers matter
With A = 0.5 V (assumed) and 50 MHz: 100 fF → −144.8 dBc/Hz (SE), −147.8 (diff); 54 fF → −142.1 / −145.1. All are ≥ 20 dB below the
reference contribution (−121.9), which is why Nagam & Kinget conclude a 100 fF sampler is negligible compared with the reference buffer.

In [ ]:
Cs = np.array([25, 54, 100, 200, 400])*1e-15
def frame(A):
    se = 10*np.log10(kT/(Cs*A**2*50e6)); di = se - 10*np.log10(2)
    return ([dict(x=Cs*1e15, y=se, mode="lines+markers", name="single-ended"), dict(x=Cs*1e15, y=di, mode="lines+markers", name="differential"),
             dict(x=[25, 400], y=[-152+30.1]*2, name="REF floor x N^2 (-121.9)", line=dict(dash="dot"))],
            f"A = {A} V at f_REF = 50 MHz: 100 fF -> {10*np.log10(kT/(100e-15*A**2*50e6)):.1f} dBc/Hz (SE), 54 fF -> {10*np.log10(kT/(54e-15*A**2*50e6)):.1f} dBc/Hz")
fig = sweep([0.2, 0.3, 0.5, 0.7, 0.9], frame, "sampled amplitude A (V)", active=2,
            layout=lambda f: f.update_xaxes(type="log", title="sampling capacitor (fF)").update_yaxes(title="in-band L at 1.6 GHz (dBc/Hz)", range=[-160, -115]))
show(fig, 420)

### Interview questions — step 49 (8)

<details><summary><b>Q1.</b> Why is sampled thermal noise kT/C, independent of R?</summary>

Noise density $4kTR$ times the noise bandwidth $1/(4RC)$ gives $kT/C$: a bigger R adds density but narrows the bandwidth by the same factor.

</details>

<details><summary><b>Q2.</b> Derive the in-band phase noise of an SSPD.</summary>

$\sigma_\phi^2 = kT/(CA^2)$ per sample, white at $f_{REF}$, held: one-sided $S_\phi = 2\sigma_\phi^2/f_{REF}$; $\mathcal L = S_\phi/2 = kT/(CA^2f_{REF})$.

</details>

<details><summary><b>Q3.</b> Why is a differential SSPD 3 dB better?</summary>

Two caps give $2kT/C$ noise variance on the difference, but the signal slope doubles (4× power), so the ratio improves by 2.

</details>

<details><summary><b>Q4.</b> What is the SSPD noise for C = 100 fF, A = 0.5 V, f_REF = 50 MHz?</summary>

$4.14\times10^{-21}/(10^{-13}\cdot0.25\cdot5\times10^7) = 3.3\times10^{-15}$ → −144.8 dBc/Hz single-ended.

</details>

<details><summary><b>Q5.</b> Why not make C very small to save area and VCO loading?</summary>

Noise rises as 1/C, and charge injection/kickback relative to C grows; also charge sharing with the hold capacitor changes gain.

</details>

<details><summary><b>Q6.</b> Besides kT/C, what other SSPD noise sources exist?</summary>

Sampling-edge (aperture) jitter of the reference buffer, the buffer/isolation amplifier noise, the transconductor noise (step 50).

</details>

<details><summary><b>Q7.</b> Why is the sampler noise not multiplied by N while aperture jitter is?</summary>

Aperture jitter is a timing error of the reference edge (×N² like reference noise); kT/C is a voltage error at the detector output, referred through the large gain A.

</details>

<details><summary><b>Q8.</b> Does a higher f_REF help the SSPD noise?</summary>

Yes, by $10\log f_{REF}$: more samples per second average the noise; also N falls, reducing the reference's N² contribution.

</details>

<a id='step-50'></a>
## Step 50 — Pulsed-Gm charge pump: why the gain is reduced, β = 2·A·g_m·τ/T_REF, noise ∝ T/τ, dump node

*Roadmap step 50: derivation*

### What happens
The held sample drives a transconductor $g_m$ whose output current is connected to the loop filter only during a short pulse
$\tau$ each reference period (Gao's "pulser"). The rest of the time the current goes to a **dump** node.

### How
Differential sample $v = 2A\sin\Delta\phi$ → current $g_m v$ for $\tau$ out of $T_{REF}$ → average
$$\overline{I} = g_m\cdot2A\sin\Delta\phi\cdot\frac{\tau}{T_{REF}}\ \Rightarrow\ \boxed{\beta = \frac{2A\,g_m\,\tau}{T_{REF}}\ [\text{A per VCO rad}]}$$
Loop design (step 36 with $I_p/2\pi N\to\beta$): $C_{tot} = \frac{b\,\beta K_{VCO}}{\omega_c^2}$. **The SSPD gain is so large that without
pulsing the filter would need nanofarads**; shrinking the duty $\tau/T$ is the cheapest way to reduce $\beta$.

**Noise.** The transconductor's noise current is gated with the same duty $d = \tau/T$: noise *power* scales with $d$, signal with $d^2$,
so the input-referred noise scales as $1/d$ — **noise ∝ T/τ**. Short pulses save area but cost noise.

**Dump node.** While the pulse is off, the $g_m$ current is steered to a node held at $V_{cont}$ (buffer), so the current source never
turns off and no charge sharing occurs when it reconnects (step 42).

In [ ]:
Kv, wc = 2*np.pi*300e6, 2*np.pi*2.5e6; A = 0.5; gm = 100e-6; fref = 50e6
bpm = np.tan(np.radians(45+50/2))
def frame(tau_ps):
    beta = 2*A*gm*tau_ps*1e-12*fref
    Ct = bpm*beta*Kv/wc**2
    duty = np.logspace(-3, 0, 200)
    return ([dict(x=duty*100, y=bpm*2*A*gm*duty*Kv/wc**2*1e12, name="C_tot (pF) for f_c = 2.5 MHz, PM 50", row=1, col=1),
             dict(x=duty*100, y=10*np.log10(1/duty), name="relative Gm noise (dB, = 10 log T/tau)", row=1, col=2),
             dict(x=[tau_ps*1e-12*fref*100], y=[Ct*1e12], mode="markers", marker=dict(size=12), name="this tau", row=1, col=1)],
            f"tau = {tau_ps} ps (duty {tau_ps*1e-12*fref*100:.2f}%), g_m = 100 uS, A = 0.5 V: beta = {beta*1e9:.0f} nA/rad, C_tot = {Ct*1e12:.1f} pF")
fig = sweep([50, 100, 200, 400, 1000, 4000, 20000], frame, "pulse width tau (ps)", active=2, rows=1, cols=2,
            subplot_titles=("loop-filter capacitance", "Gm noise penalty"), height=420,
            layout=lambda f: f.update_xaxes(type="log", title_text="duty tau/T (%)").update_yaxes(type="log", row=1, col=1))
show(fig)

### Interview questions — step 50 (10)

<details><summary><b>Q1.</b> Derive β for a pulsed-Gm charge pump.</summary>

Current $g_m\cdot2A\sin\Delta\phi$ for $\tau$ per period → average $2Ag_m(\tau/T)\Delta\phi$ for small errors → $\beta = 2Ag_m\tau/T$.

</details>

<details><summary><b>Q2.</b> Why does an SSPLL need its gain reduced?</summary>

The SSPD+Gm gain is so high that, for a few-MHz loop, $C_{tot} = b\beta K_{VCO}/\omega_c^2$ would be enormous; pulsing reduces β without changing the detector.

</details>

<details><summary><b>Q3.</b> Why does pulsing increase noise?</summary>

Signal current scales with duty d, gated noise power with d: SNR falls as d, input-referred noise rises as 1/d (∝ T/τ).

</details>

<details><summary><b>Q4.</b> Why use a dump node?</summary>

To keep the transconductor always conducting (steered, not switched off) at a node held at $V_{cont}$, avoiding turn-on transients and charge sharing at reconnection.

</details>

<details><summary><b>Q5.</b> What else does the pulse position do?</summary>

It sets the effective loop delay: a pulse right after sampling shortens the delay versus a full T/2 hold, improving phase margin (Thijssen).

</details>

<details><summary><b>Q6.</b> How does the pulse width vary over PVT and why does it matter?</summary>

An RC pulser's width follows R and C (±15–35% with R/C corners and temperature); β and the loop bandwidth follow. Our audit suggested timing the pulse from the divided VCO for a PVT-stable width.

</details>

<details><summary><b>Q7.</b> What is the old design's β and its bandwidth?</summary>

311 nA/rad with f_c = 1.16 MHz (old slides, 2.36 GHz); the new proposal targets 2.5 MHz.

</details>

<details><summary><b>Q8.</b> What sets the minimum practical τ?</summary>

Switch speed and the pulse generator's ability to produce a clean, repeatable pulse; very short pulses make β sensitive to edge shape and add noise.

</details>

<details><summary><b>Q9.</b> How do you check β in simulation?</summary>

DC/transient test: apply a known differential sample, measure average output current into a voltage source at $V_{cont}$, divide by $2A\sin\Delta\phi$ (testbench tb_sscp_dc/pulse in Phase 8).

</details>

<details><summary><b>Q10.</b> Can you reduce β by lowering g_m instead?</summary>

Yes, but a small $g_m$ means more noise per unit signal ($v_n^2 \propto 1/g_m$) and more offset; pulsing keeps $g_m$ healthy.

</details>

<a id='step-51'></a>
## Step 51 — Loop dynamics with the hold: ZOH sinc, T/2 delay, charge-sharing pole, phase-margin cost

*Roadmap step 51: Bode plots*

### What happens
The SSPD is a two-switch sampler (Nagam: sw1 onto $C_1$ at the REF edge, then sw2 shares onto $C_2$, which holds for the NCB and the
Gm). The held voltage changes once per period. Three effects add phase lag to the loop:

1. **Zero-order hold:** $\frac{1-e^{-sT}}{sT}$, magnitude $\text{sinc}(fT)$, phase $-\pi fT$ (delay $T/2$).
2. **Charge sharing** between the sampling and hold capacitors (call them $C_s$ and $C_h$ here to avoid confusion with the loop
   filter): $v_h[k] = \frac{C_s v_s[k] + C_h v_h[k-1]}{C_s+C_h}$, a first-order discrete low-pass with $\alpha = \frac{C_h}{C_s+C_h}$:
   $$H_{cs}(z) = \frac{1-\alpha}{1-\alpha z^{-1}}\quad(\alpha = 0.5\text{ for equal caps}).$$
3. **Pulse position** of the Gm (step 50).

### Why it matters
At $f_c = f_{REF}/20$ the hold delay alone costs 9°; charge sharing with equal caps adds more. The loop filter must be designed with
this extra lag (PM target at the zero/pole placement must include it), and the same lag limits the FFNC bandwidth (Phase 6).

In [ ]:
fref = 50e6; T = 1/fref; f = np.logspace(5, np.log10(fref/2), 400); z = np.exp(2j*np.pi*f*T)
def frame(alpha):
    zoh = np.sinc(f*T)*np.exp(-1j*np.pi*f*T)
    cs = (1-alpha)/(1-alpha/z)
    tot = zoh*cs
    pm_cost = -np.degrees(np.angle(tot[np.argmin(abs(f-2.5e6))]))
    return ([dict(x=f, y=20*np.log10(abs(tot)), name="|hold x charge sharing| (dB)", row=1, col=1),
             dict(x=f, y=np.degrees(np.angle(zoh)), name="ZOH phase", line=dict(dash="dot"), row=1, col=2),
             dict(x=f, y=np.degrees(np.unwrap(np.angle(tot))), name="total phase", row=1, col=2)],
            f"alpha = C_h/(C_s+C_h) = {alpha}: phase lag at 2.5 MHz = {pm_cost:.1f} deg (hold alone 9.0 deg)")
fig = sweep([0.0, 0.25, 0.5, 0.75], frame, "alpha", active=2, rows=1, cols=2, subplot_titles=("magnitude", "phase (deg)"), height=420,
            layout=lambda f: f.update_xaxes(type="log", title_text="Hz"))
show(fig)

### Interview questions — step 51 (8)

<details><summary><b>Q1.</b> What is the transfer function of a zero-order hold and its delay?</summary>

$(1-e^{-sT})/(sT)$; magnitude sinc(fT), linear phase $-\pi fT$ = a T/2 delay.

</details>

<details><summary><b>Q2.</b> Phase-margin cost of the hold at f_c = f_REF/20?</summary>

$180°\times(1/20) = 9°$.

</details>

<details><summary><b>Q3.</b> Why does Nagam use two switches (sw1/C1, sw2/C2)?</summary>

To hold the sample on a separate node (C2) before the next REF edge so the NCB and Gm see a stable voltage, isolated from the next tracking phase; it needs non-overlapping control.

</details>

<details><summary><b>Q4.</b> Derive the charge-sharing transfer function.</summary>

$v_h[k](C_s+C_h) = C_sv_s[k] + C_hv_h[k-1]$ → $H(z) = (1-\alpha)/(1-\alpha z^{-1})$, $\alpha = C_h/(C_s+C_h)$.

</details>

<details><summary><b>Q5.</b> What is the DC gain of the charge-sharing stage?</summary>

1: in steady state $v_h = v_s$ — the sharing only delays (low-pass), it does not attenuate DC.

</details>

<details><summary><b>Q6.</b> How do you reduce the hold-induced phase lag?</summary>

Smaller hold cap relative to sampling cap (α → 0), pulse the Gm early, or a buffer instead of charge sharing (costs noise/power).

</details>

<details><summary><b>Q7.</b> Does the sinc magnitude matter for the loop?</summary>

Little at $f_c \ll f_{REF}$ (sinc ≈ 1); it matters for FFNC, which works up to ~$f_{REF}/8$.

</details>

<details><summary><b>Q8.</b> Why is an SSPLL bandwidth chosen ≈ f_REF/20 rather than f_REF/10?</summary>

The hold and charge-sharing lag plus spur considerations; Nagam's analysis uses $\omega_{REF}/20$.

</details>

<a id='step-52'></a>
## Step 52 — Acquisition: periodic SSPD characteristic, false lock to N ± k, dead-zone FLL, hand-over

*Roadmap step 52: sequence and simulation*

### What happens
The SSPD output depends on the VCO phase **mod 2π at each reference edge**. Any $f_{VCO} = kf_{REF}$ gives a stable DC sample, so
the SSPD loop happily locks to $k = 31$ or $33$ (**false lock**), and its capture range is only about the loop bandwidth.

### How the dead-zone FLL fixes it
A divider (÷32), a PFD and a small charge pump compare REF with $f_{VCO}/32$. Far from lock the PFD saturates and pulls the frequency
toward $32f_{REF}$ (frequency detection). Near lock the divided phase error is inside a **dead zone** (here ±0.65π of the reference
period ≫ one VCO period $2\pi/32$), the FLL charge pump stays **off**, and the low-noise SSPD loop alone sets the phase — the
**hand-over**. Because the dead zone is wider than one VCO cycle in divider terms, there is always an SSPD lock point inside it.

### The simulation below
Cycle-by-cycle model: SSPD sampling the real phase (wrapped), pulsed Gm, third-order filter, VCO with free-running offset, and the FLL
with a PFD that slips cycles (keeping its sign) and a dead zone. Move the initial offset and compare with and without the FLL.

In [ ]:
def sspll_sim(off_ref, use_fll=True, ncyc=2500, kick_at=None, kick_V=0.0, A=0.5, beta=1.0e-6, Kv_Hz=300e6,
              fref=50e6, N=32, fc=2.5e6, pm_deg=50, Ifll=10e-6, dz=0.65*np.pi, tau=0.2e-9):
    """Cycle-by-cycle SSPLL with an optional dead-zone FLL.
    VCO free-running at (N + off_ref)*fref; SSPD samples A*sin(e) where e = VCO phase - 2*pi*N*k (wrapped);
    pulsed Gm injects -(beta*2*pi)... charge so that the average current is -beta*sin(e) per rad (small-signal beta A/rad);
    FLL: PFD between REF and VCO/N with cycle-slip memory p (rad of REF) and a dead zone |p| < dz.
    Loop filter = 3rd-order design for f_c, pm with the SSPD gain beta (A/rad of VCO phase)."""
    from scipy.linalg import expm
    T = 1/fref; Kv = 2*np.pi*Kv_Hz
    d = sl.loop_design_type2(fc, pm_deg, 2*np.pi*N*beta, Kv, N)      # same equations: I_p/(2 pi N) -> beta
    R, C1, C2 = d['R'], d['C1'], d['C2']
    Am = np.array([[-1/(R*C1), 1/(R*C1), 0], [1/(R*C2), -1/(R*C2), 0], [0, Kv, 0]])
    Bv = np.array([0, 1/C2, 0])
    def prop(dt):
        M = np.zeros((4, 4)); M[:3, :3] = Am; M[:3, 3] = Bv
        E = expm(M*dt); return E[:3, :3], E[:3, 3]
    E1, b1 = prop(tau); E2, b2 = prop(T - tau)
    x = np.zeros(3); p = 0.0; psi_prev = 0.0; fll_on = False
    out = dict(t=[], f=[], e=[], vc=[], fll=[])
    for k in range(ncyc):
        phi = 2*np.pi*(N + off_ref)*k + x[2]                 # VCO phase at the k-th REF edge (rad)
        e = np.angle(np.exp(1j*(phi - 2*np.pi*N*k)))          # wrapped VCO phase error seen by the SSPD
        i_ss = -beta*np.sin(e)*T/tau                         # pulsed Gm: average current -beta*sin(e)
        psi = phi/N - 2*np.pi*k                               # divider phase vs REF (rad of REF)
        p += psi - psi_prev; psi_prev = psi
        if p > 2*np.pi: p -= 2*np.pi                          # PFD cycle slip keeps the sign (frequency detection)
        if p < -2*np.pi: p += 2*np.pi
        i_f = 0.0
        if use_fll and abs(p) > dz:
            i_f = -np.sign(p)*Ifll*(abs(p)/(2*np.pi))*T/tau   # same charge as a PFD pulse of width |p|/2pi*T
        if kick_at is not None and k == kick_at:
            x[0] += kick_V; x[1] += kick_V                     # disturbance moves the whole filter state
        x = E1 @ x + b1*(i_ss + i_f); x = E2 @ x
        if k % 5 == 0:
            out['t'].append(k*T); out['e'].append(e); out['vc'].append(x[1]); out['fll'].append(i_f != 0)
            out['f'].append(N + off_ref + Kv_Hz*x[1]/fref)    # VCO frequency in units of f_REF
    return {k: np.array(v) for k, v in out.items()}, d

offs = [0.3, 0.7, 1.3, 2.4, 3.6]
def frame(off):
    r0, _ = sspll_sim(off, use_fll=False, ncyc=5000); r1, d = sspll_sim(off, use_fll=True, ncyc=5000)
    k0, k1 = np.round(r0['f'][-1]), np.round(r1['f'][-1])
    return ([dict(x=r0['t']*1e6, y=r0['f'], name="SSPD loop only"),
             dict(x=r1['t']*1e6, y=r1['f'], name="SSPD + dead-zone FLL"),
             dict(x=[0, r1['t'][-1]*1e6], y=[32, 32], name="target 32 f_REF", line=dict(dash="dot", color="grey"))],
            f"start {32+off:.1f} x f_REF: SSPD alone locks at {k0:.0f} x f_REF" + (" (FALSE LOCK)" if k0 != 32 else "") +
            f"; with FLL at {k1:.0f} x f_REF")
fig = sweep(offs, frame, "initial VCO offset (multiples of f_REF)", active=2,
            layout=lambda f: f.update_xaxes(title="time (us)").update_yaxes(title="VCO frequency / f_REF"))
show(fig, 440)

### Interview questions — step 52 (10)

<details><summary><b>Q1.</b> Why can an SSPLL lock to the wrong frequency?</summary>

The SSPD sees only the VCO phase at each reference edge; $f_{VCO} = kf_{REF}$ for any integer k produces constant samples.

</details>

<details><summary><b>Q2.</b> What is the capture range of an SSPD loop?</summary>

About the loop bandwidth (and at most ±$f_{REF}/2$ around the target harmonic): outside it the beat averages out.

</details>

<details><summary><b>Q3.</b> How does a dead-zone FLL work?</summary>

PFD + divider + CP pull the frequency to $Nf_{REF}$; when the divided phase error is inside the dead zone the CP is off and the SSPD loop takes over.

</details>

<details><summary><b>Q4.</b> How wide must the dead zone be?</summary>

Wider than half a VCO period referred to the divider ($\pi/N$ reference radians) so an SSPD lock point exists inside it, plus margin for transients; but small enough to catch a slip to a neighbouring harmonic.

</details>

<details><summary><b>Q5.</b> Why not keep the PFD loop on permanently?</summary>

It would add its N²-multiplied noise and CP spurs, defeating the SSPLL's purpose.

</details>

<details><summary><b>Q6.</b> What happens if the FLL dead zone is too narrow?</summary>

The FLL and SSPD fight: the FLL keeps correcting while the SSPD holds a lock point outside the dead zone → jitter and spurs, possibly no stable lock.

</details>

<details><summary><b>Q7.</b> What is hand-over?</summary>

The transition from FLL-dominated frequency acquisition to SSPD-only phase lock; designed so it happens automatically by the dead zone.

</details>

<details><summary><b>Q8.</b> Alternatives to a dead-zone FLL?</summary>

Coarse digital frequency calibration with a counter, Hsu's PFD with an auxiliary SSPD, a sub-sampling-assisted PFD, or Geng's quadrature SSPD (frequency detection from I/Q samples).

</details>

<details><summary><b>Q9.</b> How do you test false-lock immunity in simulation?</summary>

Start the VCO at several offsets (including ±1 and ±2 harmonics) and over PVT; check the final frequency is always $Nf_{REF}$ (testbench tb_top_lock, Phase 8).

</details>

<details><summary><b>Q10.</b> What is the `fll_active` test output in the proposal for?</summary>

To observe when the FLL is acting (outside its dead zone): it should be active during acquisition and quiet in lock.

</details>

<a id='step-53'></a>
## Step 53 — What the SSPLL solves (request 8)

*Roadmap step 53: N² on PD/CP, divider noise and power, CP-mismatch insensitivity, low ripple, a wider f_c for ring VCOs*

| Problem of the Type-II CP-PLL | How the SSPLL solves it | Why |
|---|---|---|
| PFD/CP noise ×N² (+30.1 dB at N = 32) | detector noise referred to VCO phase with gain A, not ×N² | no divider in the loop (step 48) |
| divider noise and power in the loop | no divider in the main loop; the FLL divider is idle in lock | SSPD samples the VCO directly |
| CP mismatch → static offset and spurs | at lock the sample ≈ 0, so the Gm current ≈ 0; no reset pulses with both sources on | the detector output itself is zero at lock |
| ripple from $I_pR_1$ steps every period | small pulsed current proportional to a tiny sample | the "error" is a voltage near zero |
| high in-band floor limits the bandwidth of a ring PLL | floor drops ~20 dB → the full $f_{REF}/20$ bandwidth suppresses ring noise | step 44 |
| — | the held voltage is a direct measure of each edge's timing error | enables FFNC (Phase 6) |

Numbers on our design point (same ring, reference, N, 2.5 MHz; models/step8_variants.py):

In [ ]:
import sys; sys.path.insert(0, "../models")
import pllmodel as pm
rows = []
for name, v, d in (("Type-II CP-PLL", "type2", pm.TYPE2_TYP), ("SSPLL", "sspll", pm.TYP), ("SSPLL + FFNC", "ffnc", pm.TYP)):
    j, f, L = pm.jitter(d, v)
    rows.append((name, j["TOTAL"]*1e12, L[np.argmin(abs(f-1e5))], L[np.argmin(abs(f-1e6))]))
fig = make_subplots(rows=1, cols=2, subplot_titles=("rms jitter 10 kHz-40 MHz (ps)", "phase noise (dBc/Hz)"))
fig.add_trace(go.Bar(x=[r[0] for r in rows], y=[r[1] for r in rows], text=[f"{r[1]:.2f}" for r in rows], name="jitter"), 1, 1)
fig.add_trace(go.Bar(x=[r[0] for r in rows], y=[r[2] for r in rows], name="L(100 kHz)"), 1, 2)
fig.add_trace(go.Bar(x=[r[0] for r in rows], y=[r[3] for r in rows], name="L(1 MHz)"), 1, 2)
fig.update_yaxes(range=[-125, -95], row=1, col=2)
show(fig, 420)
for r in rows: print(f"{r[0]:15s} jitter {r[1]:.2f} ps   L(100k) {r[2]:.1f}   L(1M) {r[3]:.1f} dBc/Hz")

### Interview questions — step 53 (9)

<details><summary><b>Q1.</b> List the problems a sub-sampling PLL solves.</summary>

No N² on PD/CP noise, no divider noise/power in the loop, insensitivity to CP mismatch (zero current at lock), low ripple/spurs from the CP, and enough in-band headroom to use a wide bandwidth with a ring VCO.

</details>

<details><summary><b>Q2.</b> Why is an SSPLL insensitive to charge-pump mismatch?</summary>

At lock the sampled voltage is ~0, so the transconductor's net output current is ~0; there are no PFD reset pulses with both sources on. Gm offset still matters (it shifts the lock point slightly).

</details>

<details><summary><b>Q3.</b> What is the in-band phase noise of a good SSPLL?</summary>

Limited by the reference ×N² and the sampler/Gm: Gao reported −126 dBc/Hz at 2.2 GHz; our budget is −120 dBc/Hz at 1.6 GHz (FFNC on).

</details>

<details><summary><b>Q4.</b> Does the SSPLL reduce reference noise?</summary>

No; reference (crystal + buffer) noise is still ×N² and becomes the in-band limit.

</details>

<details><summary><b>Q5.</b> How much does the SSPLL improve jitter for our ring VCO at the same bandwidth?</summary>

From 2.52 to 2.29 ps (typical): modest, because the ring dominates; the bigger benefit is enabling FFNC (to 1.40 ps).

</details>

<details><summary><b>Q6.</b> What does the SSPLL cost?</summary>

An acquisition loop (FLL), sampler kickback/load-modulation spurs, amplitude-dependent gain, narrow linear range, hold delay (lower usable bandwidth).

</details>

<details><summary><b>Q7.</b> Why does the SSPLL save power compared with a CP-PLL?</summary>

No high-speed divider running in lock in the main loop; the FLL divider can even be gated off after lock.

</details>

<details><summary><b>Q8.</b> Why does the SSPLL enable FFNC but a CP-PLL does not?</summary>

FFNC needs a continuous voltage proportional to the timing error of each edge; the SSPD's held sample is exactly that. A PFD gives pulse widths.

</details>

<details><summary><b>Q9.</b> What is FoM of Gao's 2.2 GHz SSPLL?</summary>

0.15 ps rms at 7.6 mW (ISSCC 2009): about −247 dB — an LC-VCO design; ring SSPLLs are around −225 to −236 dB (Nagam −236.3 with FFNC).

</details>

<a id='step-54'></a>
## Step 54 — Limit: narrow capture range, false lock, slow relock after a disturbance (Hsu, Geng)

*Roadmap step 54: explanation and simulation*

### What happens
After a disturbance (supply step, coupling burst) that moves the VCO frequency by more than the capture range, the SSPD loop alone
re-locks to the **nearest harmonic**, which may be wrong. With a dead-zone FLL it returns to $32f_{REF}$, but only after the FLL
notices (the divided phase must leave the dead zone) — **relock is slow**.

### How the literature attacks it
- **Hsu, Tripurari, Yu, Kinget (CICC 2011):** a PFD with an *auxiliary* SSPD — the PFD loop keeps acquisition robust while the SSPD lowers
  in-band noise.
- **Geng, Xie, Wang (TCAS-II 2021):** a *quadrature* SSPD samples I and Q; the pair detects the direction of frequency error, giving
  fast relock under interference without a separate FLL.

Below: lock first, then kick $V_{cont}$ at 30 µs. Second figure: the **dead-zone width** itself — too narrow and the FLL and SSPD fight
(limit cycle); wide enough and the hand-over is clean.

In [ ]:
def sspll_sim(off_ref, use_fll=True, ncyc=2500, kick_at=None, kick_V=0.0, A=0.5, beta=1.0e-6, Kv_Hz=300e6,
              fref=50e6, N=32, fc=2.5e6, pm_deg=50, Ifll=10e-6, dz=0.65*np.pi, tau=0.2e-9):
    """Cycle-by-cycle SSPLL with an optional dead-zone FLL.
    VCO free-running at (N + off_ref)*fref; SSPD samples A*sin(e) where e = VCO phase - 2*pi*N*k (wrapped);
    pulsed Gm injects -(beta*2*pi)... charge so that the average current is -beta*sin(e) per rad (small-signal beta A/rad);
    FLL: PFD between REF and VCO/N with cycle-slip memory p (rad of REF) and a dead zone |p| < dz.
    Loop filter = 3rd-order design for f_c, pm with the SSPD gain beta (A/rad of VCO phase)."""
    from scipy.linalg import expm
    T = 1/fref; Kv = 2*np.pi*Kv_Hz
    d = sl.loop_design_type2(fc, pm_deg, 2*np.pi*N*beta, Kv, N)      # same equations: I_p/(2 pi N) -> beta
    R, C1, C2 = d['R'], d['C1'], d['C2']
    Am = np.array([[-1/(R*C1), 1/(R*C1), 0], [1/(R*C2), -1/(R*C2), 0], [0, Kv, 0]])
    Bv = np.array([0, 1/C2, 0])
    def prop(dt):
        M = np.zeros((4, 4)); M[:3, :3] = Am; M[:3, 3] = Bv
        E = expm(M*dt); return E[:3, :3], E[:3, 3]
    E1, b1 = prop(tau); E2, b2 = prop(T - tau)
    x = np.zeros(3); p = 0.0; psi_prev = 0.0; fll_on = False
    out = dict(t=[], f=[], e=[], vc=[], fll=[])
    for k in range(ncyc):
        phi = 2*np.pi*(N + off_ref)*k + x[2]                 # VCO phase at the k-th REF edge (rad)
        e = np.angle(np.exp(1j*(phi - 2*np.pi*N*k)))          # wrapped VCO phase error seen by the SSPD
        i_ss = -beta*np.sin(e)*T/tau                         # pulsed Gm: average current -beta*sin(e)
        psi = phi/N - 2*np.pi*k                               # divider phase vs REF (rad of REF)
        p += psi - psi_prev; psi_prev = psi
        if p > 2*np.pi: p -= 2*np.pi                          # PFD cycle slip keeps the sign (frequency detection)
        if p < -2*np.pi: p += 2*np.pi
        i_f = 0.0
        if use_fll and abs(p) > dz:
            i_f = -np.sign(p)*Ifll*(abs(p)/(2*np.pi))*T/tau   # same charge as a PFD pulse of width |p|/2pi*T
        if kick_at is not None and k == kick_at:
            x[0] += kick_V; x[1] += kick_V                     # disturbance moves the whole filter state
        x = E1 @ x + b1*(i_ss + i_f); x = E2 @ x
        if k % 5 == 0:
            out['t'].append(k*T); out['e'].append(e); out['vc'].append(x[1]); out['fll'].append(i_f != 0)
            out['f'].append(N + off_ref + Kv_Hz*x[1]/fref)    # VCO frequency in units of f_REF
    return {k: np.array(v) for k, v in out.items()}, d

def frame(kick_mV):
    r0, _ = sspll_sim(0.0, use_fll=False, ncyc=5000, kick_at=1500, kick_V=kick_mV*1e-3)
    r1, _ = sspll_sim(0.0, use_fll=True, ncyc=5000, kick_at=1500, kick_V=kick_mV*1e-3)
    late = np.where(abs(r1['f'] - 32) > 2e-3)[0]
    tr = (r1['t'][late[-1]] - 30e-6)*1e6 if len(late) else 0
    return ([dict(x=r0['t']*1e6, y=r0['f'], name="SSPD only"), dict(x=r1['t']*1e6, y=r1['f'], name="with FLL")],
            f"V_cont kick {kick_mV} mV ({kick_mV*1e-3*300e6/50e6:.2f} f_REF): SSPD alone ends at {np.round(r0['f'][-1]):.0f} x f_REF; "
            f"with FLL at {np.round(r1['f'][-1]):.0f} x f_REF, relock {tr:.1f} us after the kick")
fig = sweep([20, 60, 120, 200, 300, 500], frame, "kick on V_cont (mV)", active=3,
            layout=lambda f: f.update_xaxes(title="time (us)").update_yaxes(title="VCO frequency / f_REF"))
show(fig, 430)
def frame2(dzp):
    r, _ = sspll_sim(2.4, use_fll=True, ncyc=6000, dz=dzp*np.pi)
    clean = r['fll'][-300:].sum() == 0 and r['f'][-300:].std() < 1e-3
    return ([dict(x=r['t']*1e6, y=r['f'], name="VCO frequency / f_REF"),
             dict(x=r['t']*1e6, y=30 + r['fll'].astype(float), name="FLL active (offset 30)", line=dict(shape="hv"))],
            f"dead zone +/-{dzp} pi (rad of REF): " + ("clean hand-over, FLL silent in lock" if clean else "FLL and SSPD fight: limit cycle"))
fig = sweep([0.3, 0.4, 0.5, 0.65, 0.8], frame2, "dead zone (x pi)", active=3,
            layout=lambda f: f.update_xaxes(title="time (us)").update_yaxes(range=[29.8, 35]))
show(fig, 430)

### Interview questions — step 54 (9)

<details><summary><b>Q1.</b> Why is the relock of an SSPLL slow?</summary>

The SSPD cannot detect frequency; the FLL only acts after the divided phase error leaves its dead zone, then must pull the frequency back, then hand over again.

</details>

<details><summary><b>Q2.</b> What happens if a disturbance moves the VCO by one f_REF and the FLL is off?</summary>

The SSPD locks to the neighbouring harmonic (31 or 33 × f_REF) — a false lock that persists.

</details>

<details><summary><b>Q3.</b> What did Hsu et al. propose?</summary>

A PFD-based PLL with an auxiliary sub-sampling PD to suppress in-band noise while keeping the PFD's robust acquisition.

</details>

<details><summary><b>Q4.</b> What is a quadrature SSPD and why does it help?</summary>

Two samplers 90° apart (I/Q); the rotation direction of the (I,Q) vector reveals the sign of the frequency error, so frequency can be corrected directly — fast relock (Geng et al.).

</details>

<details><summary><b>Q5.</b> What sets the width of the FLL dead zone?</summary>

Large enough to contain an SSPD lock point and transient phase wander (≥ π/N plus margin), small enough to catch slips to neighbouring harmonics (the divided phase moves 2π per beat).

</details>

<details><summary><b>Q6.</b> What happens with a too-narrow dead zone (seen in the simulation)?</summary>

After frequency acquisition the SSPD's lock point can sit just outside the dead zone; the FLL keeps pushing, the SSPD slips, and the two loops form a limit cycle.

</details>

<details><summary><b>Q7.</b> How do you make the FLL robust over PVT?</summary>

Choose the dead zone with margin, a modest FLL current (fast enough but not overpowering), and verify lock from many initial offsets and corners, including a mid-lock disturbance.

</details>

<details><summary><b>Q8.</b> What kind of disturbances cause relock events in an SoC?</summary>

Large supply/ground bounce from digital activity, substrate coupling bursts, power-gating events on the VCO supply.

</details>

<details><summary><b>Q9.</b> How could you detect loss of lock on chip?</summary>

Lock detector: count VCO/32 vs REF cycles over a window, or monitor whether the FLL becomes active (`fll_active` output in the proposal).

</details>

<a id='step-55'></a>
## Step 55 — Limit: sampler spurs (load modulation, charge injection, charge sharing); REF noise still ×N²; ring noise beyond f_c

*Roadmap step 55: derivations*

### What happens
The sampler is connected to the VCO only part of the time. Each reference period it (1) **changes the load** on the sampled node, (2)
**injects charge** (kickback) into it when the switch toggles, and (3) **shares charge** between the sampling and hold capacitors and the
VCO node. All three repeat at $f_{REF}$ → reference spurs.

### How (load modulation, first order)
A ring's period is proportional to the total switched node capacitance $\Sigma C$. If the sampler adds $\Delta C$ during a fraction $d$ of
each period, the VCO frequency drops by $\delta f = f_0\,\Delta C/\Sigma C$ for that fraction. The first Fourier component of this
rectangular frequency modulation is $a_1 = \frac{2\delta f}{\pi}\sin(\pi d)$, a phase modulation $\beta = a_1/f_{REF}$, and
$$\text{spur} \approx 20\log_{10}\frac{\beta}{2} = 20\log_{10}\left(\frac{f_0}{f_{REF}}\frac{\Delta C}{\Sigma C}\frac{\sin\pi d}{\pi}\right).$$
With $f_0/f_{REF} = 32$, even $\Delta C/\Sigma C = 10^{-3}$ gives about −40 dBc: an **isolation buffer** (tiny, constant input capacitance)
between VCO and sampler is mandatory, plus a dummy sampler on the complementary node to balance the load.

### The other two limits
**Reference noise** is still ×N² (−121.9 dBc/Hz in band); the REF buffer must be very low-noise. **Ring noise beyond $f_c$** is not
touched by any loop: the SSPLL alone stays at ~2.3 ps; FFNC (Phase 6) attacks exactly this band.

In [ ]:
dC = np.logspace(-3, 1, 200)*1e-15                         # added capacitance while sampling (F)
SigC = 300e-15                                              # total switched ring-node capacitance (assumed 10 nodes x 30 fF)
def frame(d):
    spur = 20*np.log10(32*dC/SigC*np.sin(np.pi*d)/np.pi)
    return ([dict(x=dC*1e15, y=spur, name=f"load modulation, duty {d}"),
             dict(x=[dC[0]*1e15, dC[-1]*1e15], y=[-45, -45], name="proposal limit -45 dBc", line=dict(dash="dot"))],
            f"track duty {d}: {np.interp(0.1e-15, dC, spur):.0f} dBc for 0.1 fF, {np.interp(1e-15, dC, spur):.0f} dBc for 1 fF of load change")
fig = sweep([0.05, 0.1, 0.25, 0.5], frame, "fraction of the period the sampler tracks", active=3,
            layout=lambda f: f.update_xaxes(type="log", title="capacitance change seen by the VCO (fF)").update_yaxes(title="spur (dBc)"))
show(fig, 420)

### Interview questions — step 55 (10)

<details><summary><b>Q1.</b> What is VCO load modulation in an SSPLL?</summary>

The sampler's capacitance appears on the VCO node only while tracking; the VCO frequency changes periodically at $f_{REF}$ → spurs.

</details>

<details><summary><b>Q2.</b> Estimate the spur from load modulation.</summary>

$\delta f/f_0 = \Delta C/\Sigma C$ for fraction d; $\beta = 2\delta f\sin(\pi d)/(\pi f_{REF})$; spur $= 20\log(\beta/2)$.

</details>

<details><summary><b>Q3.</b> Why does the spur scale with f_0/f_REF?</summary>

A given fractional frequency deviation accumulates more phase over a longer reference period: $\beta \propto \delta f/f_{REF}$.

</details>

<details><summary><b>Q4.</b> What is kickback?</summary>

Charge injected into the VCO node by the sampling switch's gate edges (feedthrough and channel charge), once per period.

</details>

<details><summary><b>Q5.</b> How do dummy samplers help?</summary>

A matched switch on the complementary node (or a half-size dummy) cancels injected charge and balances the load seen by both VCO outputs.

</details>

<details><summary><b>Q6.</b> Why use an isolation buffer between VCO and sampler?</summary>

Its input capacitance is small and constant, so the VCO sees almost no load change; the buffer absorbs the kickback. It costs power and adds its own noise.

</details>

<details><summary><b>Q7.</b> What is the 'DLL-tuned tracking edge' idea?</summary>

Track the VCO only for a short window just before the sampling edge (timed by a delay), reducing the load-modulation duty d.

</details>

<details><summary><b>Q8.</b> Which noise remains ×N² in the SSPLL?</summary>

Reference (crystal) and reference buffer noise, and sampling-edge jitter.

</details>

<details><summary><b>Q9.</b> Why does the SSPLL not fix ring noise beyond the loop bandwidth?</summary>

No loop suppresses VCO noise above $f_c$; only a cleaner VCO, more power, or feedforward cancellation helps there.

</details>

<details><summary><b>Q10.</b> What spur levels did Nagam report?</summary>

Reference spurs of −60.2 dBc without cancellation and −55.2 dBc with cancellation (the NCB path also couples the reference).

</details>

<a id='step-56'></a>
## Step 56 — Limit: sinusoidal gain non-linearity, ambiguity beyond ±π/2, ring supply sensitivity

*Roadmap step 56: explanation*

### What happens
The SSPD characteristic is $v = A\sin\Delta\phi$ (sine VCO) or a saturating curve (ring). Its slope — the loop gain — is $A\cos\Delta\phi$:
- near 0 it is maximal and linear;
- at ±π/2 the gain falls to zero (the loop stops correcting);
- beyond ±π/2 the slope is **negative**: the equilibrium there is unstable, the loop pushes the phase away → cycle slip;
- the amplitude A (or the ring's slew rate) depends on **supply, corner and temperature**, so loop bandwidth and FFNC gain move with them.

### Why it matters
Large phase errors during transients get *less* correction (slower recovery) and may slip; a supply step that changes the ring's swing
changes both $f_c$ and the cancellation gain $K_{t}$ — one reason FFNC needs calibration (Phase 6).

In [ ]:
ph = np.linspace(-np.pi, np.pi, 400)
def frame(e_deg):
    e = np.radians(e_deg)
    return ([dict(x=np.degrees(ph), y=np.sin(ph), name="SSPD characteristic sin(dphi) (x A)"),
             dict(x=np.degrees(ph), y=np.cos(ph), name="local gain cos(dphi)", line=dict(dash="dash")),
             dict(x=[e_deg], y=[np.sin(e)], mode="markers", marker=dict(size=12), name="operating point")],
            f"phase error {e_deg} deg: local gain {np.cos(e):.2f} x A" + ("  -> NEGATIVE: unstable, the loop pushes away (slip)" if np.cos(e) < 0 else ""))
fig = sweep([0, 20, 45, 70, 90, 110, 150], frame, "phase error (deg)", active=2,
            layout=lambda f: f.update_xaxes(title="VCO phase error at the sample (deg)").update_yaxes(range=[-1.2, 1.2]))
show(fig, 420)

### Interview questions — step 56 (8)

<details><summary><b>Q1.</b> What is the small-signal gain of an SSPD at phase error Δφ?</summary>

$A\cos\Delta\phi$: maximum at 0, zero at ±π/2, negative beyond.

</details>

<details><summary><b>Q2.</b> Why is there an ambiguity beyond ±π/2?</summary>

The sine is symmetric; a sample value corresponds to two phases, one on each slope. On the negative slope feedback becomes positive and the phase runs away to the next stable point.

</details>

<details><summary><b>Q3.</b> How does the ring supply affect the SSPLL loop?</summary>

The ring's swing and edge rate change with supply → $K_{PD}$ changes → loop bandwidth and FFNC gain change; supply noise also directly modulates the VCO frequency.

</details>

<details><summary><b>Q4.</b> How do you make the SSPD gain less amplitude-dependent?</summary>

Limit/saturate the waveform with a buffer (gain set by the buffer's slope), regulate the ring supply, or calibrate (Nagam's NCB gain calibration absorbs $K_t$ changes).

</details>

<details><summary><b>Q5.</b> Why might large phase transients recover slowly in an SSPLL?</summary>

The effective gain falls as cos(Δφ); at large errors the loop gain is small, so recovery is slow and may slip.

</details>

<details><summary><b>Q6.</b> Is the PFD characteristic better in this respect?</summary>

Yes: linear over ±2π with constant slope, and frequency detection beyond; the SSPD trades this robustness for low noise.

</details>

<details><summary><b>Q7.</b> What is the linear range in time for our 1.6 GHz VCO?</summary>

±π/2 VCO radians ≈ ±156 ps; for <10% gain error, about ±0.45 rad ≈ ±45 ps.

</details>

<details><summary><b>Q8.</b> How does the ring waveform shape change the characteristic?</summary>

A sharper (more square) waveform gives a steeper slope near the crossing (higher gain) but a narrower linear range, saturating sooner.

</details>

<a id='step-57'></a>
## Step 57 — Fixes: dummy samplers, isolation buffers, tracking window, low-noise REF buffer, combined PD, QSSPD, FFNC

*Roadmap step 57: limit-to-fix table*

| Limit | Fix | Where in our design |
|---|---|---|
| false lock / narrow capture | dead-zone FLL (÷32 + PFD + CP) | FLL block, `fll_en`, `fll_active` |
| slow relock | quadrature SSPD (Geng) or PFD + auxiliary SSPD (Hsu) | future scope; FLL with tuned dead zone now |
| load modulation | isolation buffer, short tracking window (delay-tuned) | AC-coupled buffers on ring nodes |
| kickback / charge injection | dummy sampler on the complementary node, differential sampling | differential SSPD with dummies |
| charge sharing / hold lag | sized $C_s/C_h$, non-overlapping clocks, early Gm pulse | two-switch SSPD (Nagam), clock generator |
| filter area from high gain | pulsed Gm ($\beta = 2Ag_m\tau/T$) | pulsed-Gm CP, RC/derived pulser |
| amplitude-dependent gain | limit the swing, current-fed (CCO) ring, calibration | CCO supply; NCB gain calibration |
| REF noise ×N² | low-noise, steep-edge REF buffer; clean crystal (≤ −155 dBc/Hz floor) | self-biased 3.3 V receiver |
| ring noise beyond $f_c$ | **feedforward noise cancellation** | NCB delay line, `ffnc_en`, `ncb_code` |

### Interview questions — step 57 (7)

<details><summary><b>Q1.</b> Name three ways to reduce SSPLL reference spurs.</summary>

Isolation buffers between VCO and sampler, dummy samplers/differential sampling, a short tracking window (low duty), and careful layout symmetry.

</details>

<details><summary><b>Q2.</b> Why must the reference buffer be very low-noise in an SSPLL?</summary>

Reference noise is ×N² and, with the detector noise removed from N², it is the dominant in-band source.

</details>

<details><summary><b>Q3.</b> What does a combined PFD/SSPD give you?</summary>

Robust acquisition and frequency detection from the PFD with in-band noise from the SSPD (Hsu).

</details>

<details><summary><b>Q4.</b> What problem does FFNC solve that no loop technique can?</summary>

VCO noise between the loop bandwidth and ~$f_{REF}/8$, where the loop cannot help.

</details>

<details><summary><b>Q5.</b> Why does a CCO (current-fed ring) help the SSPLL?</summary>

Better supply rejection (current source output resistance) → less supply-induced phase noise and gain variation; also lower supply-path noise than an LDO in our study.

</details>

<details><summary><b>Q6.</b> Which fixes cost power?</summary>

Isolation buffers, low-noise REF buffer, FFNC delay line and its amplifier; the proposal budgets about 0.8 mW for the NCB.

</details>

<details><summary><b>Q7.</b> How does a non-overlapping clock generator help the two-switch SSPD?</summary>

It guarantees sw2 opens/closes only when sw1 is open, so the hold node never connects directly to the tracking node.

</details>

<a id='step-58'></a>
## Step 58 — Lab: SSPLL lock, explained waveform by waveform

*Roadmap step 58: annotated plots*

### What you do
Run the full behavioural SSPLL from different coarse-calibration errors and read every waveform:
1. **VCO frequency** — FLL drags it toward $32f_{REF}$ (straight ramps = PFD saturated), then the SSPD takes over.
2. **SSPD sample** $\sin(e)$ — a beat while out of lock; it freezes near 0 at lock.
3. **FLL activity** — on during acquisition, off in lock (dead zone).
4. **$V_{cont}$** — settles at the value that cancels the coarse error: $\Delta V = -\text{offset}\cdot f_{REF}/K_{VCO}$.

The transistor-level testbenches of the previous design (`tb_top_lock`, `tb_top_spur`, `tb_sscp_chain` in the project, 2.36 GHz) show
the same sequence; the new 1.6 GHz benches are written in Phase 8 and must reproduce these waveforms.

In [ ]:
def sspll_sim(off_ref, use_fll=True, ncyc=2500, kick_at=None, kick_V=0.0, A=0.5, beta=1.0e-6, Kv_Hz=300e6,
              fref=50e6, N=32, fc=2.5e6, pm_deg=50, Ifll=10e-6, dz=0.65*np.pi, tau=0.2e-9):
    """Cycle-by-cycle SSPLL with an optional dead-zone FLL.
    VCO free-running at (N + off_ref)*fref; SSPD samples A*sin(e) where e = VCO phase - 2*pi*N*k (wrapped);
    pulsed Gm injects -(beta*2*pi)... charge so that the average current is -beta*sin(e) per rad (small-signal beta A/rad);
    FLL: PFD between REF and VCO/N with cycle-slip memory p (rad of REF) and a dead zone |p| < dz.
    Loop filter = 3rd-order design for f_c, pm with the SSPD gain beta (A/rad of VCO phase)."""
    from scipy.linalg import expm
    T = 1/fref; Kv = 2*np.pi*Kv_Hz
    d = sl.loop_design_type2(fc, pm_deg, 2*np.pi*N*beta, Kv, N)      # same equations: I_p/(2 pi N) -> beta
    R, C1, C2 = d['R'], d['C1'], d['C2']
    Am = np.array([[-1/(R*C1), 1/(R*C1), 0], [1/(R*C2), -1/(R*C2), 0], [0, Kv, 0]])
    Bv = np.array([0, 1/C2, 0])
    def prop(dt):
        M = np.zeros((4, 4)); M[:3, :3] = Am; M[:3, 3] = Bv
        E = expm(M*dt); return E[:3, :3], E[:3, 3]
    E1, b1 = prop(tau); E2, b2 = prop(T - tau)
    x = np.zeros(3); p = 0.0; psi_prev = 0.0; fll_on = False
    out = dict(t=[], f=[], e=[], vc=[], fll=[])
    for k in range(ncyc):
        phi = 2*np.pi*(N + off_ref)*k + x[2]                 # VCO phase at the k-th REF edge (rad)
        e = np.angle(np.exp(1j*(phi - 2*np.pi*N*k)))          # wrapped VCO phase error seen by the SSPD
        i_ss = -beta*np.sin(e)*T/tau                         # pulsed Gm: average current -beta*sin(e)
        psi = phi/N - 2*np.pi*k                               # divider phase vs REF (rad of REF)
        p += psi - psi_prev; psi_prev = psi
        if p > 2*np.pi: p -= 2*np.pi                          # PFD cycle slip keeps the sign (frequency detection)
        if p < -2*np.pi: p += 2*np.pi
        i_f = 0.0
        if use_fll and abs(p) > dz:
            i_f = -np.sign(p)*Ifll*(abs(p)/(2*np.pi))*T/tau   # same charge as a PFD pulse of width |p|/2pi*T
        if kick_at is not None and k == kick_at:
            x[0] += kick_V; x[1] += kick_V                     # disturbance moves the whole filter state
        x = E1 @ x + b1*(i_ss + i_f); x = E2 @ x
        if k % 5 == 0:
            out['t'].append(k*T); out['e'].append(e); out['vc'].append(x[1]); out['fll'].append(i_f != 0)
            out['f'].append(N + off_ref + Kv_Hz*x[1]/fref)    # VCO frequency in units of f_REF
    return {k: np.array(v) for k, v in out.items()}, d

def frame(off):
    r, d = sspll_sim(off, use_fll=True, ncyc=4000)
    late = np.where(abs(r['f'] - 32) > 2e-3)[0]
    tl = r['t'][late[-1]]*1e6 if len(late) else 0
    return ([dict(x=r['t']*1e6, y=r['f'], name="f_VCO / f_REF", row=1, col=1),
             dict(x=r['t']*1e6, y=np.sin(r['e']), name="SSPD sample sin(e)", row=2, col=1),
             dict(x=r['t']*1e6, y=r['fll'].astype(float), name="FLL active", line=dict(shape="hv"), row=3, col=1),
             dict(x=r['t']*1e6, y=r['vc']*1e3, name="V_cont (mV)", row=4, col=1)],
            f"coarse error {off:+.1f} f_REF ({off*50:.0f} MHz): locked at 32 x f_REF after {tl:.1f} us; final V_cont {r['vc'][-1]*1e3:.0f} mV "
            f"(expected {-off*50e6/300e6*1e3:.0f} mV)")
fig = sweep([-2.0, -0.8, 0.4, 1.0, 1.8, 3.0], frame, "coarse frequency error (multiples of f_REF)", active=3, rows=4, cols=1,
            shared_xaxes=True, subplot_titles=("VCO frequency / f_REF", "SSPD sample", "FLL active", "V_cont (mV)"), height=800,
            vertical_spacing=0.06, layout=lambda f: f.update_xaxes(title_text="time (us)", row=4, col=1))
show(fig)

### Interview questions — step 58 (7)

<details><summary><b>Q1.</b> Describe the lock sequence of our SSPLL.</summary>

Coarse code sets the CCO near 1.6 GHz; the FLL (÷32 + PFD) pulls the frequency to 32 f_REF; its divided phase error enters the dead zone and the FLL goes quiet; the SSPD loop locks the phase to the nearest VCO zero crossing; FFNC can then be enabled.

</details>

<details><summary><b>Q2.</b> What does a beating SSPD sample tell you?</summary>

The VCO is not at an integer multiple of f_REF; the beat frequency is the distance to the nearest harmonic.

</details>

<details><summary><b>Q3.</b> How do you confirm in simulation that the PLL locked to the right harmonic?</summary>

Measure the VCO frequency over many reference periods (or count VCO/32 vs REF edges): it must be exactly 32 × f_REF; the FLL must be inactive.

</details>

<details><summary><b>Q4.</b> What final V_cont do you expect for a coarse error of +1 f_REF with K_VCO = 300 MHz/V?</summary>

−50 MHz / 300 MHz/V = −167 mV relative to the free-running point.

</details>

<details><summary><b>Q5.</b> Why is lock time longer for larger coarse errors?</summary>

The FLL slews the frequency at a limited rate (its CP current into the filter), and each harmonic crossing takes time before the SSPD can hold.

</details>

<details><summary><b>Q6.</b> What would you check in the transistor-level lock bench besides lock?</summary>

Ripple on V_cont, spurs (FFT), FLL hand-over timing, power, behaviour over PVT and Monte Carlo, and that the ring supply node stays below 1.8 V.

</details>

<details><summary><b>Q7.</b> Why are behavioural results checked before transistor sims?</summary>

They are fast, expose architecture-level issues (dead-zone width, FLL strength, loop stability) and give targets for the circuits.

</details>